In [1]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.spatial.transform import Rotation as R
import scipy.integrate as integrate

In [2]:
fname_hnodes = "hnodes.txt"
fnames_out = ["D1_am_start.conf", "D1_mdna_start.conf"]

fname_dat = ["D1_am.dat", "D1_mdna.dat"]

am_inds = {0: list(range(2733, 3275+1))[::-1],
           1: list(range(2189, 2732+1)),
           2: list(range(1643, 2188+1))[::-1],
           3: list(range(1096, 1642+1)),
           4: list(range(547, 1095+1))[::-1],
           5: list(range(0, 546+1)),
           6: list(range(7117, 7666+1))[::-1],
           7: list(range(6569, 7116+1)),
           8: list(range(6017, 6568+1))[::-1],
           9: list(range(5465, 6016+1)),
          10: list(range(4916, 5464+1))[::-1],
          11: list(range(4368, 4915+1)),
          12: list(range(3821, 4367+1))[::-1],
          13: list(range(3276, 3820+1))}

mdna_inds = {0: list(range(4910, 5457+1))[::-1],
             1: list(range(5458, 6015+1)),
             2: list(range(6016, 6573+1))[::-1],
             3: list(range(6574, 7131+1)),
             4: list(range(7132, 7689+1))[::-1],
             5: list(range(7690, 7731+1)) + list(range(0, 508+1)),
             6: list(range(509, 1063+1))[::-1],
             7: list(range(1064, 1615+1)),
             8: list(range(1616, 2170+1))[::-1],
             9: list(range(2171, 2721+1)),
             10: list(range(2722, 3265+1))[::-1],
             11: list(range(3266, 3813+1)),
             12: list(range(3814, 4361+1))[::-1],
             13: list(range(4362, 4909+1))}

inds = [am_inds, mdna_inds]

In [3]:
def read_nucleotides(fname_conf):
    with open(fname_conf, "r") as f:
        lines = f.readlines()
        lines = lines[0+3: len(lines)]
        nts = np.zeros((len(lines), 3*3))
        for i, line in enumerate(lines):
            vec = line.split(" ")
            if len(vec) >= 9:
                nts[i, :] = vec[0:3*3]
            else:
                print("Information missing for nucleotide {}. Check configuration file.\n".format(i))
        com = np.mean(nts[:, 0:3], axis=0)
        nts[:, 0:3] = nts[:, 0:3] - np.tile(com, (np.shape(nts)[0], 1))
        return nts


def solve_splines(x, y):
    count = len(x)
    a = np.copy(y)
    b = np.zeros(count - 1)
    d = np.zeros(count - 1)
    h = np.zeros(count - 1)
    alpha = np.zeros(count - 1)
    for i in range(0, count - 1):
        h[i] = x[i + 1] - x[i]
    for i in range(1, count - 1):
        alpha[i] = (3 / h[i]) * (a[i + 1] - a[i]) - (3 / h[i - 1]) * (a[i] - a[i - 1])
    c = np.zeros(count)
    l = np.zeros(count)
    l[0] = 1
    mu = np.zeros(count)
    z = np.zeros(count)
    for i in range(1, count - 1):
        l[i] = 2 * (x[i + 1] - x[i - 1]) - h[i - 1] * mu[i - 1]
        mu[i] = h[i] / l[i]
        z[i] = (alpha[i] - h[i - 1] * z[i - 1]) / l[i]
    l[count - 1] = 1
    for j in range(count - 2, -1, -1):
        c[j] = z[j] - mu[j] * c[j + 1]
        b[j] = (a[j + 1] - a[j]) / h[j] - (h[j] * (c[j + 1] + 2 * c[j])) / 3
        d[j] = (c[j + 1] - c[j]) / (3 * h[j])
    return np.stack((a[0:count - 1], b[0:count - 1], c[0:count - 1], d[0:count - 1], x[0:count - 1]))


def spline_point(t_p, P_x, P_y, P_z, t):
    for i in range(len(P_x[0, :])):
        if P_x[4, i] <= t_p <= t[i+1]:
            x = P_x[0, i] + P_x[1, i]*(t_p - P_x[4, i]) + P_x[2, i]*(t_p - P_x[4, i])**2 + P_x[3, i]*(t_p - P_x[4, i])**3
            y = P_y[0, i] + P_y[1, i]*(t_p - P_y[4, i]) + P_y[2, i]*(t_p - P_y[4, i])**2 + P_y[3, i]*(t_p - P_y[4, i])**3
            z = P_z[0, i] + P_z[1, i]*(t_p - P_z[4, i]) + P_z[2, i]*(t_p - P_z[4, i])**2 + P_z[3, i]*(t_p - P_z[4, i])**3
            return np.array([x, y, z])

        
def spline_derivative(t_p, P_x, P_y, P_z, t):
    for i in range(len(P_x[0, :])):
        if P_x[4, i] <= t_p <= t[i+1]:
            x = P_x[1, i] + 2*P_x[2, i]*(t_p - P_x[4, i]) + 3*P_x[3, i]*(t_p - P_x[4, i])**2
            y = P_y[1, i] + 2*P_y[2, i]*(t_p - P_y[4, i]) + 3*P_y[3, i]*(t_p - P_y[4, i])**2
            z = P_z[1, i] + 2*P_z[2, i]*(t_p - P_z[4, i]) + 3*P_z[3, i]*(t_p - P_z[4, i])**2
            return np.sqrt(x**2 + y**2 + z**2)

        
def construct_splines(nodes):
    splines = {}
    for h, node_array in nodes.items():
        t = np.linspace(0, 1, node_array.shape[0])
        Px = solve_splines(t, node_array[:, 0])
        Py = solve_splines(t, node_array[:, 1])
        Pz = solve_splines(t, node_array[:, 2])
        splines[h] = [t, Px, Py, Pz]
    return splines


def read_hnodes(fname):
    hnodes = {}
    hnodes_arrays = {}
    with open(fname, 'r') as file:
        lines = file.readlines()
        i = 0
        j = 0
        while i < len(lines) - 1 and j < len(lines) - 1:
            for i in range(j, len(lines)):
                coords = lines[i].split(",")
                if len(coords) == 1 and int(coords[0]) not in list(hnodes):
                    h_num = int(coords[0])
                    hnodes[h_num] = []
                    break
            for j in range(i+1, len(lines)):
                coords = lines[j].split(",")
                if len(coords) == 3:
                    hnodes[h_num].append([float(coords[0]), float(coords[1]), float(coords[2])])
                else:
                    break
        for h_num, coords in hnodes.items():
            array = np.zeros((len(coords), 3))
            for i, point in enumerate(coords):
                array[i, 0] = point[0]
                array[i, 1] = point[1]
                array[i, 2] = (-1)*point[2] # left-handed vs. right-handed coordinate systems
            hnodes_arrays[h_num] = array
    return hnodes_arrays


def sample_from_spline_uniformly(spline, n):
    t = spline[0]
    Px = spline[1]
    Py = spline[2]
    Pz = spline[3]
    array = np.zeros((n, 3))
    l_tot, _ = integrate.quad(spline_derivative, 0, 1, args=(Px, Py, Pz, t), limit=150)
    count = 0
    for i in range(1, len(t)):
        l_seg, _ = integrate.quad(spline_derivative, t[i-1], t[i], args=(Px, Py, Pz, t))
        n_points = int(np.round((l_seg / l_tot) * n))
        if i == len(t)-1 and n_points + count < n:
            n_points += n - n_points - count
        t_p = np.linspace(t[i-1], t[i], n_points, endpoint=False)
        for j in range(n_points):
            if count + j < n:
                array[count + j, :] = spline_point(t_p[j], Px, Py, Pz, t)
        count += n_points
    return array


def get_target_hnodes(fname, min_l):
    hnodes = read_hnodes(fname)
    splines = construct_splines(hnodes)
    pos_dct = sample_from_splines(splines, min_l)
    # center by subtracting mean
    mean_pos = np.zeros((1, 3))
    for h_num, pos in pos_dct.items():
        mean_pos += np.sum(pos_dct[h_num], axis=0) * (1/min_l)
    mean_pos = mean_pos / len(pos_dct)
    for h_num, pos in pos_dct.items():
        pos_dct[h_num] = pos - mean_pos
    return pos_dct


def add_nucleotides(pos, bp, inds, pairs, h):
    if h % 2 == 0:
        phase = 0
    else:
        phase = np.pi
    offset = (140/360)*2*np.pi
    d = 0.6 / np.sin((70/360)*2*np.pi)
    if inds[1] >= inds[0]:
        sign = +1
    else:
        sign = -1
    for i in range(np.shape(bp)[0]):
        if i == np.shape(bp)[0]-1:
            T = bp[i, :] - bp[i-1, :]
        else:
            T = bp[i+1, :] - bp[i, :]
        T_hat = T / np.linalg.norm(T)
        rot, _ = R.align_vectors(T_hat, np.array([1, 0, 0]))
        rot.as_matrix()
        N_hat = rot.apply(np.array([0, 1, 0]))
        B_hat = rot.apply(np.array([0, 0, 1]))
        r1 = d*np.cos(-phase)*N_hat - d*np.sin(-phase)*B_hat
        r2 = d*np.cos(-phase-offset)*N_hat - d*np.sin(-phase-offset)*B_hat
        p1 = bp[i, :] + r1
        p2 = bp[i, :] + r2
        pos.append([inds[i], p1, -r1/np.linalg.norm(r1), sign*T_hat])
        try:
            pair = pairs[inds[i]]
            pos.append([pair, p2, -r2/np.linalg.norm(r2), (-1)*sign*T_hat])
        except KeyError:
            pass
        phase += 2*np.pi / 10.50
        
        
def solve_pairs(nts):
    pairs = {}
    # print("Looking for pairs...")
    for i in range(np.shape(nts)[0]):
        # hb pos is at r_base_nt + A*e1 + B*e2 + C*e3, using the reference frame of base nt
        A = 1.2
        B = 0
        C = 0
        e1 = nts[i, 3:6]
        e3 = nts[i, 6:9]
        e2 = np.linalg.cross(e3, e1)
        hb_pos = nts[i, 0:3] + A*e1 + B*e2 + C*e3
        for j in range(i+1, np.shape(nts)[0]): 
            d = np.sqrt(np.sum(np.power(hb_pos - nts[j, 0:3], 2)))
            if d <= 0.1:
                # print("Found pair {}-{}!".format(i, j))
                pairs[i] = j
                pairs[j] = i
                break
    return pairs
                    

def write_target_conf(fname_hnodes, fname_out, inds, nts):
    box = 750
    hnodes = read_hnodes(fname_hnodes)
    splines = construct_splines(hnodes)
    pairs = solve_pairs(nts)
    positions = []
    for h, inds in inds.items():
        bp_centers = sample_from_spline_uniformly(splines[h], len(inds))
        add_nucleotides(positions, bp_centers, inds, pairs, h)
    positions = sorted(positions, key=lambda pos: pos[0])
    print(len(positions))
    print(len(pairs))
    with open(fname_out, "w") as f:
        f.write("t = 0\n")
        f.write("b = {} {} {}\n".format(box, box, box))
        f.write("E = 0 0 0\n")
        for pos in positions:
            line = "{} {} {}".format(pos[1][0], pos[1][1], pos[1][2])
            line += " {} {} {}".format(pos[2][0], pos[2][1], pos[2][2])
            line += " {} {} {}".format(pos[3][0], pos[3][1], pos[3][2])
            line += " {} {} {} {} {} {}\n".format(0, 0, 0, 0, 0, 0)
            f.write(line)

In [4]:
for i, fname_out in enumerate(fnames_out):
    nts = read_nucleotides(fname_dat[i])
    write_target_conf(fname_hnodes, fname_out, inds[i], nts)

15306
15278
15436
15408
